<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">

# The Data Scientist
## Book 2 · Python Data Analysis, Visualization, and Storytelling
### Notebook 04 · Basic SQL Awareness
© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br> The Python Quants GmbH | https://tpq.io<br>

https://thedatascientist.dev | https://linktr.ee/dyjh

### Why this notebook exists
The point of this notebook is to keep relational thinking visible: select, filter,
group, and join. The SQL helper is loaded directly from the book local `code/` folder
so the path mechanics stay explicit.

This cell makes the notebook portable. It finds the book root locally, or clones
`tdscode` in Colab, so the later paths and imports work without manual edits.

This cell makes the notebook portable. It finds the book root locally, or clones
`tdscode` in Colab, so the later paths and imports work without manual edits.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "2_data"
REPO_URL = "https://github.com/yhilpisch/tdscode"

cwd = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in (cwd, *cwd.parents):
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")
    if not repo_root.exists():
        # Clone the companion repo once in Colab.
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
    BOOK_ROOT = repo_root / BOOK_NAME

os.chdir(BOOK_ROOT)

# Make the book root and the helper folder importable.
for path in (BOOK_ROOT, BOOK_ROOT / "code"):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

requirements = BOOK_ROOT / "requirements.txt"
if requirements.exists() and "google.colab" in sys.modules:
    # Keep Colab aligned with the book dependencies.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )

print("Book root:", BOOK_ROOT)


In [ ]:
from pathlib import Path
import importlib.util

module_path = BOOK_ROOT / "code" / "sql_demo.py"
spec = importlib.util.spec_from_file_location("sql_demo", module_path)
sql_demo = importlib.util.module_from_spec(spec)
assert spec and spec.loader is not None
spec.loader.exec_module(sql_demo)

build_demo = sql_demo.build_demo

conn, cur, orders, customers = build_demo()

print(orders)
print()
print(customers)


This cell loads `sql_demo.py` from the book-local `code/` folder and builds a tiny
in-memory database. That keeps the SQL examples executable without extra setup.

In [ ]:
sql_select = '''
SELECT
    order_id,
    amount
FROM orders;
'''

sql_filter = '''
SELECT
    order_id,
    amount
FROM orders
WHERE amount >= 100.0;
'''

print(cur.execute(sql_select).fetchall())
print()
print(cur.execute(sql_filter).fetchall())


This cell shows a basic `SELECT` and `WHERE` query. The output is the same kind of
filtered list the learner can already make in pandas.

In [ ]:
sql_group = '''
SELECT
    c.country,
    SUM(o.amount) AS total_amount
FROM orders AS o
JOIN customers AS c
    ON o.customer_id = c.customer_id
GROUP BY
    c.country
ORDER BY
    total_amount DESC;
'''

sql_result = cur.execute(sql_group).fetchall()
print(sql_result)

pandas_result = (
    orders.merge(customers, on='customer_id', how='left')
    .groupby('country')['amount']
    .sum()
    .sort_values(ascending=False)
)
print()
print(pandas_result)


This cell compares a SQL join/group-by query with the equivalent pandas aggregation.
Seeing both side by side makes the table logic easier to transfer between tools.

In [ ]:
artifact_dir = BOOK_ROOT / 'artifacts'
artifact_dir.mkdir(parents=True, exist_ok=True)

note_path = artifact_dir / 'sql_awareness_note.txt'
note_path.write_text(
    'SQL and pandas express the same table logic in different syntax.\n'
    'SELECT matches column selection, WHERE matches filtering, GROUP BY\n'
    'matches aggregation, and JOIN matches merge.\n'
)

conn.close()
print(note_path.resolve())


<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">